# 01 Data Cleaning

This notebook implements the cleaning decisions identified during the data quality and duplicate audits.

The goal is to create a cleaner and more consistent dataset before performing exploratory analysis, feature engineering, and recommendation modelling.

Rather than immediately removing records, each potential cleaning issue is first inspected and quantified. Cleaning rules are only applied after their impact on the dataset has been evaluated.

## 1. Load Data

The cleaned dataset will be created from the original Spotify tables stored in DuckDB.

<!-- Before applying any modifications, all relevant tables are loaded into memory. -->

In [2]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config.paths import DUCKDB_PATH

con = duckdb.connect(DUCKDB_PATH)

print(f"Connected to DuckDB database: {DUCKDB_PATH}")

Connected to DuckDB database: C:\Users\hp\OneDrive - WU Wien\Desktop\Denis_Masters\Semester 2\Solution Engineering - Python\Spotify_Project\spotify_project\data\interim\spotify.duckdb


In [3]:
# Load tables

tracks_df = con.execute(
    "SELECT * FROM tracks"
).df()

artists_df = con.execute(
    "SELECT * FROM artists"
).df()

albums_df = con.execute(
    "SELECT * FROM albums"
).df()

audio_features_df = con.execute(
    "SELECT * FROM audio_features"
).df()

lyrics_features_df = con.execute(
    "SELECT * FROM lyrics_features"
).df()

print("Loaded datasets:")
print(f"Tracks: {len(tracks_df):,}")
print(f"Artists: {len(artists_df):,}")
print(f"Albums: {len(albums_df):,}")
print(f"Audio Features: {len(audio_features_df):,}")
print(f"Lyrics Features: {len(lyrics_features_df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded datasets:
Tracks: 101,939
Artists: 56,129
Albums: 75,511
Audio Features: 101,909
Lyrics Features: 94,954


## 2. Cleaning Candidates Overview

Previous audits identified several potential data quality issues.

The following section summarizes all candidates for cleaning before individual decisions are made.

In [4]:
cleaning_candidates = pd.DataFrame({
    "Issue": [
        "Export artifact columns",
        "Constant columns",
        "Artists without genres",
        "Tracks without lyrics features",
        "Failed lyrics feature extraction",
        "Unknown time signatures",
        "Potential duplicate song versions",
        "Non-musical content"
    ],
    "Count": [
        5,
        3,
        23540,
        6985,
        15534,
        367,
        8548,
        1108
    ]
})

cleaning_candidates

,Issue,Count
0,Export artifact columns,5
1,Constant columns,3
2,Artists without genres,23540
3,Tracks without lyrics features,6985
4,Failed lyrics feature extraction,15534
5,Unknown time signatures,367
6,Potential duplicate song versions,8548
7,Non-musical content,1108


## 3. Export Artifact Columns

Several tables contain automatically generated index columns originating from previous exports.

These columns do not contain meaningful information and can be removed safely.

In [5]:
artifact_columns = {
    "tracks": ["column00"],
    "artists": ["column0"],
    "albums": ["column00"],
    "audio_features": ["column000"],
    "lyrics_features": ["column0"]
}

for table, cols in artifact_columns.items():
    print(f"{table}: {cols}")

tracks: ['column00']
artists: ['column0']
albums: ['column00']
audio_features: ['column000']
lyrics_features: ['column0']


## 4. Constant Columns

Columns containing only a single unique value do not contribute useful information and increase dataset size unnecessarily.

In [6]:
constant_columns = [
    ("tracks", "type"),
    ("artists", "type"),
    ("albums", "type")
]

pd.DataFrame(constant_columns, columns=["table", "column"])

,table,column
0,tracks,type
1,artists,type
2,albums,type


## 5. Lyrics Quality Issues

Lyrics-based analyses require valid lyric information.

Rows with failed lyric feature extraction are inspected separately before deciding whether they should be removed or excluded from later analyses.

In [7]:
lyrics_df = lyrics_features_df.copy()

invalid_lyrics = (
    lyrics_df["n_words"] == -1
)

print("Invalid lyric feature rows:")
print(invalid_lyrics.sum())

print("Percentage:")
print(round(invalid_lyrics.mean()*100,2))

Invalid lyric feature rows:
15534
Percentage:
16.36


In [8]:
valid_lyrics = lyrics_df[
    lyrics_df["n_words"] != -1
]

print(len(valid_lyrics))

79420


## 6. Missing Genre Information

Genre coverage is incomplete for a large share of artists.

Before removing anything, we evaluate how much of the dataset would be affected.

In [9]:
artists_df["missing_genres"] = (
    artists_df["genres"]
    .astype(str)
    .str.strip()
    .eq("[]")
)

artists_df["missing_genres"].value_counts()

missing_genres
False    32589
True     23540
Name: count, dtype: int64

## 7. Unknown Time Signatures

A small number of tracks contain a time signature value of 0.

These records are inspected before deciding whether replacement or removal is necessary.

In [10]:
tracks_df["time_signature"].value_counts().sort_index()

time_signature
0.0      367
1.0     1175
3.0    10030
4.0    88020
5.0     2347
Name: count, dtype: int64

## 8. Alternative Versions of Songs

The duplicate audit identified many tracks labelled as remasters, remixes, live recordings, radio edits, acoustic versions, and similar variations.

These records may represent the same underlying song and can introduce redundancy into recommendation systems.

In [11]:
version_patterns = {
    "remaster": r"remaster|remastered",
    "live": r"\blive\b|live at|live from|live in",
    "acoustic": r"acoustic",
    "remix": r"remix|rework|edit",
    "radio_edit": r"radio edit",
    "deluxe": r"deluxe",
    "instrumental": r"instrumental",
    "karaoke": r"karaoke",
    "demo": r"\bdemo\b",
    "anniversary": r"anniversary"
}

version_summary = []

for label, pattern in version_patterns.items():

    count = (
        tracks_df["name"]
        .str.lower()
        .str.contains(pattern, regex=True, na=False)
        .sum()
    )

    version_summary.append({
        "version_type": label,
        "track_count": count,
        "share_pct": round(count / len(tracks_df) * 100, 2)
    })

version_summary_df = pd.DataFrame(version_summary)

version_summary_df.sort_values(
    "track_count",
    ascending=False
)

,version_type,track_count,share_pct
3,remix,3364,3.30
0,remaster,1003,0.98
1,live,830,0.81
4,radio_edit,699,0.69
2,acoustic,621,0.61
6,instrumental,182,0.18
8,demo,32,0.03
9,anniversary,9,0.01
7,karaoke,5,0.00
5,deluxe,1,0.00


## 9. Non-Musical Content Detection

The duration audit revealed that the dataset contains audiobooks, lectures, white-noise recordings, educational content, children's stories, and other spoken-word material.

Since the project focuses on music recommendation, these records may need special treatment.

In [12]:
non_music_patterns = {
    "episode": r"\bepisode\b",
    "chapter": r"\bchapter\b",
    "lecture": r"lecture",
    "story": r"story|stories",
    "audiobook": r"audiobook",
    "sleep": r"sleep",
    "white_noise": r"white noise",
    "rain": r"\brain\b",
    "ocean": r"ocean",
    "waves": r"waves",
    "meditation": r"meditation",
    "myth": r"myth",
    "buddhism": r"buddhism",
    "lesson": r"lesson",
    "nouns": r"nouns",
    "verbs": r"verbs"
}

non_music_summary = []

for label, pattern in non_music_patterns.items():

    count = (
        tracks_df["name"]
        .str.lower()
        .str.contains(pattern, regex=True, na=False)
        .sum()
    )

    non_music_summary.append({
        "content_type": label,
        "track_count": count
    })

non_music_summary_df = pd.DataFrame(non_music_summary)

non_music_summary_df.sort_values(
    "track_count",
    ascending=False
)

,content_type,track_count
1,chapter,512
7,rain,440
5,sleep,260
13,lesson,248
15,verbs,198
3,story,139
8,ocean,131
9,waves,110
14,nouns,79
11,myth,46


## Over 15min Threshold - Long Tracks

In [13]:
long_tracks = tracks_df[
    tracks_df["duration_ms"] > 15*60*1000
]

In [14]:
long_tracks[["name","duration_ms"]]

,name,duration_ms
31,The Story of Alice and the White Rabbit / The ...,956547.0
38,Dracula's Guest,1684770.0
52,Banning Words,1114920.0
53,A Little Cloud,1763056.0
229,Appalachian Spring Suite,1504800.0
...,...,...
101483,Manfred: Part I,1285000.0
101485,Manfred: Part II,1616608.0
101751,Nouns 1,1323960.0
101753,Verbs 1,1357453.0


## Data Cleaning Assessment

This notebook evaluated the main data quality issues identified during the previous data audit and duplicate analysis stages. The goal was to determine which records and attributes should be retained, modified, or removed before continuing with exploratory analysis and recommendation modelling.

### Structural Data Quality

The dataset is structurally well organized. No exact duplicate rows were found in any of the available tables, and all identifier columns were unique. This indicates that the data collection and integration process did not introduce technical duplicates or duplicated Spotify identifiers.

Several tables contained automatically generated export columns (`column0`, `column00`, and `column000`) that do not contain meaningful information. In addition, the `type` columns in the tracks, artists, and albums tables were constant across all observations and therefore provide no analytical value. These columns can be removed safely without affecting the dataset.

### Lyrics Data Quality

The lyrics-related data revealed one of the most important quality issues in the project. Although lyric features were available for 94,954 tracks, further inspection showed that 15,534 records contained placeholder values (`-1`) across all lyric feature columns.

These rows do not represent valid lyric information and most likely indicate failed lyric feature extraction. After excluding these invalid records, the effective lyric feature coverage decreases from 93.15% to 77.91% of all tracks. Because these records still correspond to valid songs, they should not be removed from the dataset, but they should be excluded from any lyric-based analysis or recommendation component.

### Genre Information

Genre information was unavailable for 23,540 artists, corresponding to approximately 41.94% of the artist dataset. However, this issue reflects incomplete metadata rather than invalid records. Removing these artists would significantly reduce dataset coverage and introduce additional bias. Therefore, artists without genre information should be retained.

### Time Signature Values

A small number of tracks (367 records) contained a time signature value of 0. These values likely represent unknown or unavailable time signatures rather than invalid observations. Since they account for only a very small proportion of the dataset, they are not expected to meaningfully influence the analysis and should therefore be retained.

### Alternative Versions of Songs

The duplicate analysis revealed a substantial number of alternative song versions. The dataset contains remixes, remasters, live recordings, acoustic versions, radio edits, instrumentals, demos, and anniversary editions.

Although these tracks introduce redundancy, they still represent legitimate musical recordings available on Spotify. Removing them would artificially reduce catalog diversity and potentially eliminate versions that users genuinely listen to. Consequently, these records should remain in the dataset at this stage.

### Non-Musical Content

The most significant cleaning opportunity identified during the audit concerns non-musical content.

Keyword-based inspection and duration analysis revealed the presence of educational recordings, language-learning material, spoken-word content, audiobooks, stories, lectures, meditation recordings, sleep sounds, white-noise tracks, and other non-musical audio content.

Additional evidence was provided by the duration audit, which identified 1,108 tracks exceeding 15 minutes in length. Manual inspection confirmed that many of these recordings are audiobook chapters, educational lessons, spoken-word recordings, children's stories, and similar content types rather than songs.

Because the objective of this project is to develop a music recommender system, these records do not align with the intended recommendation domain. Their inclusion could distort exploratory analyses, influence feature distributions, bias similarity calculations, and generate irrelevant recommendations.

For this reason, non-musical content represents the strongest candidate for removal during the cleaning phase.

### Cleaning Decisions

Based on the findings of this notebook, the following cleaning decisions are proposed:

| Issue                             | Decision                          |
| --------------------------------- | --------------------------------- |
| Export artifact columns           | Remove                            |
| Constant columns                  | Remove                            |
| Invalid lyric feature rows (`-1`) | Exclude from lyric-based analyses |
| Missing genre information         | Keep                              |
| Unknown time signatures (`0`)     | Keep                              |
| Alternative song versions         | Keep                              |
| Non-musical content               | Remove                            |

After applying these cleaning decisions, the resulting dataset should provide a cleaner and more representative foundation for exploratory analysis, feature engineering, and recommendation modelling.


In [15]:
non_music_patterns = [
    r"\bchapter\b",
    r"\blesson\b",
    r"\bnouns\b",
    r"\bverbs\b",
    r"\bstory\b",
    r"\bstories\b",
    r"\blecture\b",
    r"\bepisode\b"
]

pattern = "|".join(non_music_patterns)

tracks_df["suspected_non_music"] = (
    tracks_df["name"]
    .str.lower()
    .str.contains(pattern, regex=True, na=False)
)

print("Tracks flagged as non-music:")
print(tracks_df["suspected_non_music"].sum())

print("Percentage:")
print(
    round(
        tracks_df["suspected_non_music"].mean() * 100,
        2
    )
)

Tracks flagged as non-music:
1064
Percentage:
1.04


In [16]:
tracks_df.loc[
    tracks_df["suspected_non_music"],
    ["name", "duration_ms"]
].sample(50, random_state=42)

,name,duration_ms
5171,"Nightfall (Episode Of Dimension X, September 2...",1740199.0
101781,Present Perfect Weak Verbs - I,168347.0
35480,"Persuasion: Chapter 7, Shocking Truths",993600.0
73669,Chapter 4 in Spanish,268000.0
76901,Chapter 31,454252.0
55334,White Fang - Chapter 12,332480.0
58217,Moby Dick read by Hayward Morse - Chapter 1,392093.0
58648,"Dr Jeckyll & Mr Hyde: Chapter 8, Into The Abyss",1547240.0
7697,Lesson 12 -Very Big Numbers,261533.0
35471,"Persuasion: Chapter 4, At The Seaside",1001373.0


In [17]:
cleaning_impact = pd.DataFrame({
    "Stage": [
        "Original tracks",
        "After removing non-music"
    ],
    "Tracks": [
        len(tracks_df),
        len(tracks_df) - tracks_df["suspected_non_music"].sum()
    ]
})

cleaning_impact["Remaining (%)"] = (
    cleaning_impact["Tracks"]
    / cleaning_impact.loc[0, "Tracks"]
    * 100
).round(2)

cleaning_impact

,Stage,Tracks,Remaining (%)
0,Original tracks,101939,100.00
1,After removing non-music,100875,98.96


In [18]:
tracks_df["suspected_non_music"].sum()

1064

In [19]:
tracks_non_music = tracks_df[
    tracks_df["suspected_non_music"]
].copy()

In [20]:
tracks_non_music.to_csv(
    "removed_non_music_tracks.csv",
    index=False
)

In [21]:
print(len(tracks_non_music))

1064


In [22]:
tracks_non_music[
    ["name", "duration_ms"]
].head(20)

,name,duration_ms
29,The Ketchup Story,341280.0
31,The Story of Alice and the White Rabbit / The ...,956547.0
48,The Child's Story by Charles Dickens,736092.0
389,Envelopes (Chapter VI),329293.0
888,The Little Red Hen (Story),558933.0
1010,Brazilian Portuguese Language Lesson 1,725931.0
1015,Lesson 2,256653.0
1020,Lesson 5,357538.0
1025,Lesson 12,301767.0
1034,Verbs 3,666773.0


In [23]:
tracks_non_music.sample(
    50,
    random_state=42
)[["name", "duration_ms"]]

,name,duration_ms
5171,"Nightfall (Episode Of Dimension X, September 2...",1740199.0
101781,Present Perfect Weak Verbs - I,168347.0
35480,"Persuasion: Chapter 7, Shocking Truths",993600.0
73669,Chapter 4 in Spanish,268000.0
76901,Chapter 31,454252.0
55334,White Fang - Chapter 12,332480.0
58217,Moby Dick read by Hayward Morse - Chapter 1,392093.0
58648,"Dr Jeckyll & Mr Hyde: Chapter 8, Into The Abyss",1547240.0
7697,Lesson 12 -Very Big Numbers,261533.0
35471,"Persuasion: Chapter 4, At The Seaside",1001373.0


In [24]:
tracks_non_music[
    ["name"]
].sample(
    100,
    random_state=42
)

,name
5171,"Nightfall (Episode Of Dimension X, September 2..."
101781,Present Perfect Weak Verbs - I
35480,"Persuasion: Chapter 7, Shocking Truths"
73669,Chapter 4 in Spanish
76901,Chapter 31
55334,White Fang - Chapter 12
58217,Moby Dick read by Hayward Morse - Chapter 1
58648,"Dr Jeckyll & Mr Hyde: Chapter 8, Into The Abyss"
7697,Lesson 12 -Very Big Numbers
35471,"Persuasion: Chapter 4, At The Seaside"


#### To be even more sure, another test

In [25]:
non_music_patterns = [
    r"\bchapter\b",
    r"\bchapter\s+\d+\b",
    r"\bchapter\s+[ivxlcdm]+\b",
    r"\bepisode\b",
    r"\blanguage lesson\b",
    r"\blesson\s+\d+\b",
    r"\bnouns\s+\d+\b",
    r"\bverbs\s+\d+\b",
    r"read by",
    r"\bpresent tense\b",
    r"\bpast tense\b",
    r"\bpresent perfect\b",
    r"\bweak verbs\b",
    r"\bpossessive adjectives\b",
    r"\bdirect object pronouns\b"
]

pattern = "|".join(non_music_patterns)

tracks_df["suspected_non_music"] = (
    tracks_df["name"]
    .str.lower()
    .str.contains(pattern, regex=True, na=False)
)

print("Tracks flagged as non-music:")
print(tracks_df["suspected_non_music"].sum())

print("Percentage:")
print(round(tracks_df["suspected_non_music"].mean() * 100, 2))

Tracks flagged as non-music:
950
Percentage:
0.93


In [26]:
tracks_df.loc[
    tracks_df["suspected_non_music"],
    ["name", "duration_ms"]
].sample(50, random_state=42)

,name,duration_ms
20553,"The Hound Of The Baskervilles: Chapter 6, The ...",1269867.0
101749,German Language Lesson 7,334739.0
76893,Chapter 29,254520.0
89339,The Whisperer In Darkness - Chapter 5,1104427.0
95839,Verbs 2,603133.0
62653,AR Verbs Present Tense Olvidar - Recordar,343413.0
90698,Lesson 62 - Using Direct Object Pronouns in Se...,358000.0
101500,Present Perfect Auxiliary Verbs,394333.0
55334,White Fang - Chapter 12,332480.0
14675,"Wuthering Heights: Chapter 1, A New Tenant",1013600.0


## Playlist-Based Non-Music Detection

Track names alone may create false positives, because some real songs contain words such as "story" or "lesson".

To improve the detection of non-musical content, playlist names are also inspected. If a track belongs to playlists related to language learning, audiobooks, comedy, stories, sleep sounds, meditation, or white noise, it is more likely to fall outside the scope of a music recommender.

In [27]:
playlist_non_music_patterns = [
    r"learn",
    r"lesson",
    r"language",
    r"german",
    r"spanish",
    r"french",
    r"english",
    r"comedy",
    r"joke",
    r"podcast",
    r"audiobook",
    r"audio book",
    r"story",
    r"stories",
    r"sleep",
    r"white noise",
    r"rain",
    r"ocean",
    r"waves",
    r"meditation",
    r"relaxation",
    r"relaxing",
    r"ambient sounds"
]

playlist_pattern = "|".join(playlist_non_music_patterns)

tracks_df["playlist_suspected_non_music"] = (
    tracks_df["playlist"]
    .astype(str)
    .str.lower()
    .str.contains(playlist_pattern, regex=True, na=False)
)

print("Tracks flagged by playlist name:")
print(tracks_df["playlist_suspected_non_music"].sum())

print("Percentage:")
print(round(tracks_df["playlist_suspected_non_music"].mean() * 100, 2))

Tracks flagged by playlist name:
3068
Percentage:
3.01


In [28]:
track_non_music_patterns = [
    r"\bchapter\b",
    r"\bchapter\s+\d+\b",
    r"\bchapter\s+[ivxlcdm]+\b",
    r"\bepisode\b",
    r"\blanguage lesson\b",
    r"\blesson\s+\d+\b",
    r"\bnouns\s+\d+\b",
    r"\bverbs\s+\d+\b",
    r"read by",
    r"\bpresent tense\b",
    r"\bpast tense\b",
    r"\bpresent perfect\b",
    r"\bweak verbs\b",
    r"\bpossessive adjectives\b",
    r"\bdirect object pronouns\b"
]

track_pattern = "|".join(track_non_music_patterns)

tracks_df["track_suspected_non_music"] = (
    tracks_df["name"]
    .astype(str)
    .str.lower()
    .str.contains(track_pattern, regex=True, na=False)
)

tracks_df["suspected_non_music"] = (
    tracks_df["track_suspected_non_music"] |
    tracks_df["playlist_suspected_non_music"]
)

print("Tracks flagged by track name:")
print(tracks_df["track_suspected_non_music"].sum())

print("Tracks flagged by playlist name:")
print(tracks_df["playlist_suspected_non_music"].sum())

print("Tracks flagged overall:")
print(tracks_df["suspected_non_music"].sum())

print("Overall percentage:")
print(round(tracks_df["suspected_non_music"].mean() * 100, 2))

Tracks flagged by track name:
950
Tracks flagged by playlist name:
3068
Tracks flagged overall:
3563
Overall percentage:
3.5


In [29]:
non_music_flag_summary = pd.DataFrame({
    "Flag Type": [
        "Track name only",
        "Playlist name only",
        "Both track and playlist",
        "Total flagged"
    ],
    "Count": [
        ((tracks_df["track_suspected_non_music"]) & (~tracks_df["playlist_suspected_non_music"])).sum(),
        ((~tracks_df["track_suspected_non_music"]) & (tracks_df["playlist_suspected_non_music"])).sum(),
        ((tracks_df["track_suspected_non_music"]) & (tracks_df["playlist_suspected_non_music"])).sum(),
        tracks_df["suspected_non_music"].sum()
    ]
})

non_music_flag_summary

,Flag Type,Count
0,Track name only,495
1,Playlist name only,2613
2,Both track and playlist,455
3,Total flagged,3563


In [30]:
tracks_df.loc[
    tracks_df["suspected_non_music"],
    ["name", "playlist", "duration_ms"]
].sample(50, random_state=42)

,name,playlist,duration_ms
61648,Chapter 16,Science Fiction,321213.0
6275,Stop Us - Radio Edit,French Indie Pop,206112.0
80210,Chapter 6 - Kundalini: The Light At The End Of...,Eastern Spirituality,106000.0
37780,Paper Cutting Part 2,ASMR Sleep Sounds,123630.0
83975,Bath Tub,Comedy Goes Country,15013.0
21670,Lesson 47 - More Infinitives,Learn Russian,574907.0
6862,That's It (I'm Crazy),NTC: High Intensity Training Tracks,204538.0
6261,Rooftops,NTC: High Intensity Training Tracks,177465.0
39653,"Thunder & Rain Sounds, Pt. 83",Sleep Sounds,65649.0
24981,A Night In The Jungle With Birds At River (Fluss),The Sleep Machine: Rainforest,316418.0


#### Now stricer playlist rules

In [31]:
playlist_non_music_patterns = [
    r"\blearn\s+(german|spanish|french|english|russian|chinese|portuguese|swedish)\b",
    r"\blanguage lesson\b",
    r"\bcomedy\b",
    r"\bjokes?\b",
    r"\baudiobook\b",
    r"\baudio book\b",
    r"\bshort stories\b",
    r"\bscary stories\b",
    r"\banimal stories\b",
    r"\bscience fiction\b",
    r"\bfiction\b",
    r"\bjane austen\b",
    r"\brussian lit\b",
    r"\basmr\b",
    r"\bwhite noise\b",
    r"\bsleep sounds\b",
    r"\bguided meditation\b"
]

playlist_pattern = "|".join(playlist_non_music_patterns)

tracks_df["playlist_suspected_non_music"] = (
    tracks_df["playlist"]
    .astype(str)
    .str.lower()
    .str.contains(playlist_pattern, regex=True, na=False)
)

C:\Users\hp\AppData\Local\Temp\ipykernel_3492\1219628746.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(playlist_pattern, regex=True, na=False)


In [32]:
track_non_music_patterns = [
    r"\bchapter\b",
    r"\bchapter\s+\d+\b",
    r"\bchapter\s+[ivxlcdm]+\b",
    r"\bepisode\b",
    r"\blanguage lesson\b",
    r"\blesson\s+\d+\b",
    r"\bnouns\s+\d+\b",
    r"\bverbs\s+\d+\b",
    r"read by",
    r"\bpresent tense\b",
    r"\bpast tense\b",
    r"\bpresent perfect\b",
    r"\bweak verbs\b",
    r"\bpossessive adjectives\b",
    r"\bdirect object pronouns\b"
]

track_pattern = "|".join(track_non_music_patterns)

tracks_df["track_suspected_non_music"] = (
    tracks_df["name"]
    .astype(str)
    .str.lower()
    .str.contains(track_pattern, regex=True, na=False)
)

In [33]:
tracks_df["suspected_non_music"] = (
    tracks_df["track_suspected_non_music"] |
    tracks_df["playlist_suspected_non_music"]
)

non_music_flag_summary = pd.DataFrame({
    "Flag Type": [
        "Track name only",
        "Playlist name only",
        "Both track and playlist",
        "Total flagged"
    ],
    "Count": [
        ((tracks_df["track_suspected_non_music"]) & (~tracks_df["playlist_suspected_non_music"])).sum(),
        ((~tracks_df["track_suspected_non_music"]) & (tracks_df["playlist_suspected_non_music"])).sum(),
        ((tracks_df["track_suspected_non_music"]) & (tracks_df["playlist_suspected_non_music"])).sum(),
        tracks_df["suspected_non_music"].sum()
    ]
})

non_music_flag_summary

,Flag Type,Count
0,Track name only,322
1,Playlist name only,1239
2,Both track and playlist,628
3,Total flagged,2189


In [34]:
tracks_df.loc[
    tracks_df["suspected_non_music"],
    ["name", "playlist", "duration_ms"]
].sample(60, random_state=42)

,name,playlist,duration_ms
11869,Lesson 6,Learn Arabic,1252023.0
78990,Expressions 3,Learn Spanish,742240.0
95794,Bridge to A-Ok,Comedy Top Tracks,123322.0
16355,My Little Dog,Animal Stories,27960.0
85374,A Transgression,Russian Lit,719734.0
34794,Italian Aria (From Persuasion 1995 Film),Jane Austen,85926.0
4685,Greenroom: Rachel and Amy,Women of Comedy,69976.0
18399,Call Me At __o'clock,Learn Swedish,4373.0
82994,Friendly Tollbooth,Comedy Top Tracks,203747.0
51941,Bowling,Comedy Top Tracks,310960.0


In [35]:
cleaning_impact = pd.DataFrame({
    "Stage": ["Original tracks", "After removing suspected non-music"],
    "Tracks": [
        len(tracks_df),
        len(tracks_df) - tracks_df["suspected_non_music"].sum()
    ]
})

cleaning_impact["Remaining (%)"] = (
    cleaning_impact["Tracks"] / len(tracks_df) * 100
).round(2)

cleaning_impact

,Stage,Tracks,Remaining (%)
0,Original tracks,101939,100.00
1,After removing suspected non-music,99750,97.85


## Non-Music Removal Decision

After combining track-title indicators with playlist-title indicators, 2,189 tracks were flagged as suspected non-musical content.

Manual inspection of the flagged sample showed that most of these records are language lessons, audiobook chapters, spoken-word recordings, comedy tracks, ASMR recordings, white-noise tracks, sleep sounds, or radio drama content.

Since the goal of this project is to build a music recommender, these records are outside the intended scope of the dataset. Removing them improves the relevance of the dataset while retaining 97.85% of the original track records.

# Final Cleaning Actions

Based on the data quality audit, duplicate analysis, and non-musical content inspection, the following cleaning actions will be applied to the dataset before exploratory analysis and recommendation modelling.

---

## 1. Remove Export Artifact Columns

Several tables contain automatically generated index columns originating from previous exports. These columns do not contain meaningful information and do not contribute to any analysis.

### Columns to Remove

| Table | Column |
|---------|---------|
| tracks | column00 |
| artists | column0 |
| albums | column00 |
| audio_features | column000 |
| lyrics_features | column0 |

### Reason

These columns are export artifacts and contain no useful analytical information.

---

## 2. Remove Constant Columns

The `type` column was found to contain only a single value within each table.

### Columns to Remove

| Table | Column |
|---------|---------|
| tracks | type |
| artists | type |
| albums | type |

### Reason

These columns contain no variation and therefore provide no information for analysis, visualization, or modelling.

---

## 3. Remove Non-Musical Content

A combined detection approach using track titles and playlist names identified records that likely represent non-musical audio content.

### Examples of Removed Content

- Audiobook chapters
- Language-learning lessons
- Grammar exercises
- Spoken-word recordings
- Educational recordings
- Radio dramas
- Comedy recordings
- White-noise recordings
- Sleep sounds
- ASMR recordings
- Fiction readings
- Story collections

### Examples Identified During Inspection

- Chapter 54
- White Fang – Chapter 6
- Lesson 55 – Using the "Shi" and "Shi... De" Construction
- Expressions 3
- Verbs 1
- Nouns 4
- Madame Bovary: Chapter 6
- Anna Karenina: Chapter 6
- White Noise – BP 228 Hz
- Thunder & Rain Sounds
- Cleansed Episode 0

### Removal Impact

| Metric | Value |
|---------|---------:|
| Original tracks | 101,939 |
| Removed non-musical tracks | 2,189 |
| Remaining tracks | 99,750 |
| Remaining dataset (%) | 97.85% |

### Reason

The objective of this project is to develop a music recommender system. These records do not represent music and would introduce noise into exploratory analysis, similarity calculations, feature distributions, and recommendation results.

---

## 4. Invalid Lyrics Features

The lyrics feature dataset contains rows where all extracted lyric features failed and were replaced with placeholder values (`-1`).

### Affected Records

| Metric | Value |
|---------|---------:|
| Invalid lyric feature rows | 15,534 |
| Percentage of lyrics feature dataset | 16.36% |

### Action

These tracks will remain in the dataset.

However, invalid lyric feature rows will be excluded whenever lyric-based analyses or lyric-based recommendation features are used.

### Reason

The tracks themselves are valid songs. Only the lyric feature extraction process failed.

---

## Items That Will NOT Be Removed

The following issues were identified during the audit but will not result in record removal.

### Missing Genre Information

| Metric | Value |
|---------|---------:|
| Artists without genres | 23,540 |
| Percentage of artists | 41.94% |

Reason:

Genre information is incomplete metadata rather than invalid data. Removing these artists would unnecessarily reduce dataset coverage.

---

### Unknown Time Signatures

| Metric | Value |
|---------|---------:|
| Tracks with time_signature = 0 | 367 |
| Percentage of tracks | 0.36% |

Reason:

These values likely represent unknown or unavailable time signatures and do not significantly affect the dataset.

---

### Alternative Song Versions

Examples:

- Remixes
- Remasters
- Live recordings
- Acoustic versions
- Radio edits
- Instrumental versions

Reason:

Although these tracks introduce redundancy, they are legitimate Spotify music tracks and remain part of the recommendation catalog.

---

## Expected Dataset After Cleaning

| Dataset | Before | After |
|---------|---------:|---------:|
| Tracks | 101,939 | 99,750 |
| Artists | 56,129 | 56,129 |
| Albums | 75,511 | 75,511 |
| Audio Features | 101,909 | To be updated after track filtering |
| Lyrics Features | 94,954 | To be updated after track filtering |

The cleaning process removes clearly irrelevant content while preserving the vast majority of musical recordings available in the dataset.

## Apply Cleaning Decisions

The following section applies the cleaning decisions defined above.

The original raw tables are not modified. Instead, cleaned copies are created and saved in the `data/processed` folder.

In [36]:
from pathlib import Path

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Processed data will be saved to: {processed_dir}")

Processed data will be saved to: c:\Users\hp\OneDrive - WU Wien\Desktop\Denis_Masters\Semester 2\Solution Engineering - Python\Spotify_Project\spotify_project\data\processed


In [37]:
# Create cleaned copies

tracks_clean = tracks_df.copy()
artists_clean = artists_df.copy()
albums_clean = albums_df.copy()
audio_features_clean = audio_features_df.copy()
lyrics_features_clean = lyrics_features_df.copy()

In [38]:
# Drop export artifact columns and constant columns

columns_to_drop = {
    "tracks": ["column00", "type"],
    "artists": ["column0", "type"],
    "albums": ["column00", "type"],
    "audio_features": ["column000"],
    "lyrics_features": ["column0"]
}

tracks_clean = tracks_clean.drop(columns=columns_to_drop["tracks"], errors="ignore")
artists_clean = artists_clean.drop(columns=columns_to_drop["artists"], errors="ignore")
albums_clean = albums_clean.drop(columns=columns_to_drop["albums"], errors="ignore")
audio_features_clean = audio_features_clean.drop(columns=columns_to_drop["audio_features"], errors="ignore")
lyrics_features_clean = lyrics_features_clean.drop(columns=columns_to_drop["lyrics_features"], errors="ignore")

In [39]:
# Save removed non-music tracks before filtering

removed_non_music_tracks = tracks_clean[
    tracks_clean["suspected_non_music"]
].copy()

removed_non_music_tracks.to_csv(
    processed_dir / "removed_non_music_tracks.csv",
    index=False
)

print(f"Saved removed non-music tracks: {len(removed_non_music_tracks):,}")

Saved removed non-music tracks: 2,189


In [40]:
# Remove suspected non-music tracks from main tracks table

tracks_clean = tracks_clean[
    ~tracks_clean["suspected_non_music"]
].copy()

print(f"Tracks after removing non-music content: {len(tracks_clean):,}")

Tracks after removing non-music content: 99,750


In [41]:
# Remove helper flag columns from final clean tracks table

helper_cols = [
    "track_suspected_non_music",
    "playlist_suspected_non_music",
    "suspected_non_music"
]

tracks_clean = tracks_clean.drop(columns=helper_cols, errors="ignore")

In [42]:
# Filter audio and lyric features to match the cleaned tracks table

valid_track_ids = set(tracks_clean["id"])

audio_features_clean = audio_features_clean[
    audio_features_clean["track_id"].isin(valid_track_ids)
].copy()

lyrics_features_clean = lyrics_features_clean[
    lyrics_features_clean["track_id"].isin(valid_track_ids)
].copy()

print(f"Audio features after filtering: {len(audio_features_clean):,}")
print(f"Lyrics features after filtering: {len(lyrics_features_clean):,}")

Audio features after filtering: 99,720
Lyrics features after filtering: 92,840


In [43]:
# Create valid lyrics features table for lyric-based analysis

lyrics_features_valid_clean = lyrics_features_clean[
    lyrics_features_clean["n_words"] != -1
].copy()

print(f"Valid lyric feature rows after cleaning: {len(lyrics_features_valid_clean):,}")

Valid lyric feature rows after cleaning: 77,545


In [44]:
# Save cleaned datasets

tracks_clean.to_parquet(processed_dir / "tracks_clean.parquet", index=False)
artists_clean.to_parquet(processed_dir / "artists_clean.parquet", index=False)
albums_clean.to_parquet(processed_dir / "albums_clean.parquet", index=False)
audio_features_clean.to_parquet(processed_dir / "audio_features_clean.parquet", index=False)
lyrics_features_clean.to_parquet(processed_dir / "lyrics_features_clean.parquet", index=False)
lyrics_features_valid_clean.to_parquet(processed_dir / "lyrics_features_valid_clean.parquet", index=False)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [49]:
cleaning_summary = pd.DataFrame({
    "Dataset": [
        "tracks",
        "artists",
        "albums",
        "audio_features",
        "lyrics_features",
        "lyrics_features (valid only)"
    ],
    "Original Rows": [
        len(tracks_df),
        len(artists_df),
        len(albums_df),
        len(audio_features_df),
        len(lyrics_features_df),
        len(lyrics_features_df)
    ],
    "Cleaned Rows": [
        len(tracks_clean),
        len(artists_clean),
        len(albums_clean),
        len(audio_features_clean),
        len(lyrics_features_clean),
        len(lyrics_features_valid_clean)
    ]
})

cleaning_summary["Rows Removed"] = (
    cleaning_summary["Original Rows"]
    - cleaning_summary["Cleaned Rows"]
)

cleaning_summary["Retention (%)"] = (
    cleaning_summary["Cleaned Rows"]
    / cleaning_summary["Original Rows"]
    * 100
).round(2)

cleaning_summary

,Dataset,Original Rows,Cleaned Rows,Rows Removed,Retention (%)
0,tracks,101939,99750,2189,97.85
1,artists,56129,56129,0,100.00
2,albums,75511,75511,0,100.00
3,audio_features,101909,99720,2189,97.85
4,lyrics_features,94954,92840,2114,97.77
5,lyrics_features (valid only),94954,77545,17409,81.67


The cleaned track dataset contains 99,750 records after removing suspected non-musical content.

The related audio and lyric feature tables were filtered to keep only records belonging to the cleaned track set. A separate valid lyrics feature table was also created, excluding rows where lyric feature extraction failed and all lyric features were stored as `-1`.

The raw data remains unchanged, while the cleaned outputs are stored in `data/processed`.

## Dataset Dimensions Before and After Cleaning

The table below summarizes how the dimensions of each dataset changed after the cleaning process.

In [50]:
shape_summary = pd.DataFrame({
    "Dataset": [
        "tracks",
        "artists",
        "albums",
        "audio_features",
        "lyrics_features"
    ],
    "Original Columns": [
        tracks_df.shape[1],
        artists_df.shape[1],
        albums_df.shape[1],
        audio_features_df.shape[1],
        lyrics_features_df.shape[1]
    ],
    "Cleaned Columns": [
        tracks_clean.shape[1],
        artists_clean.shape[1],
        albums_clean.shape[1],
        audio_features_clean.shape[1],
        lyrics_features_clean.shape[1]
    ]
})

shape_summary

,Dataset,Original Columns,Cleaned Columns
0,tracks,35,30
1,artists,10,8
2,albums,16,14
3,audio_features,209,208
4,lyrics_features,8,7


In [51]:
cleaning_metadata = {
    "original_tracks": len(tracks_df),
    "clean_tracks": len(tracks_clean),
    "removed_non_music_tracks": len(removed_non_music_tracks),
    "original_lyrics": len(lyrics_features_df),
    "valid_lyrics": len(lyrics_features_valid_clean)
}

pd.DataFrame(
    cleaning_metadata.items(),
    columns=["Metric", "Value"]
).to_csv(
    processed_dir / "cleaning_metadata.csv",
    index=False
)

In [52]:
print("Processed datasets saved:")

for file in sorted(processed_dir.glob("*clean*.parquet")):
    print(file.name)

Processed datasets saved:
albums_clean.parquet
artists_clean.parquet
audio_features_clean.parquet
lyrics_features_clean.parquet
lyrics_features_valid_clean.parquet
tracks_clean.parquet


## Cleaning Summary

The cleaning process focused on removing non-informative columns, identifying invalid lyric feature records, and filtering non-musical content from the dataset.

A total of 2,189 tracks were removed because they were identified as audiobook chapters, language-learning lessons, spoken-word recordings, comedy content, ASMR recordings, sleep sounds, or other non-musical material. This reduced the track dataset from 101,939 to 99,750 records while retaining 97.85% of the original catalog.

Export artifact columns and constant columns were removed from all relevant tables. Invalid lyric feature rows were preserved in the original lyrics dataset but a separate valid lyric feature dataset was created for future lyric-based analyses.

The resulting processed datasets provide a cleaner and more consistent foundation for exploratory analysis, feature engineering, and recommendation modelling.

In [53]:
con.close()